# Modèle FT-Transformer


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import zipfile
import os
import pandas as pd
import copy
import ast

## Configuration

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device : {DEVICE}")
D_EMB = 192
N_LAYERS = 3
N_HEADS = 8
FFN_FACTOR = 4/3
ATTN_DROPOUT = 0.2
FFN_DROPOUT = 0.1
RESIDUAL_DROPOUT = 0.0
BATCH_SIZE = 64      
LR = 5e-4             
WEIGHT_DECAY = 5e-5
N_EPOCHS = 80
PATIENCE = 15       

Utilisation du device : cpu


## 1. Chargement des fichiers

In [ ]:
ZIP_FILE_PATH = 'data/data.zip'
print(f"Lecture des fichiers depuis l'archive : {ZIP_FILE_PATH}")
with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zf:
    try:
        train_df = pd.read_csv(zf.open('train.csv'))
        test_df = pd.read_csv(zf.open('test.csv'))
    except KeyError:
        train_df = pd.read_csv(zf.open(os.path.join('data', 'train.csv')))
        test_df = pd.read_csv(zf.open(os.path.join('data', 'test.csv')))
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

Lecture des fichiers depuis l'archive : data/data.zip


## 2. Pre-processing

In [ ]:
TARGETS = ['satisfaction', 'wip', 'investissement']
X = train_df.drop(columns=['id'] + TARGETS, errors='ignore')
y = train_df[TARGETS]
X_test_final = test_df.drop(columns=['id'], errors='ignore')
test_ids = test_df['id']


with open("top_400_features.txt", "r", encoding="utf-8") as f:
    top_features_list = ast.literal_eval(f.read())
        
valid_features = [f for f in top_features_list if f in X.columns]
print(f"Features demandées : {len(top_features_list)}")
print(f"Features trouvées et conservées : {len(valid_features)}")
X = X[valid_features]
X_test_final = X_test_final[valid_features]  

cat_cols = [c for c in X.columns if X[c].dtype == 'object' or X[c].dtype == 'bool' or (X[c].nunique() < 20)]
num_cols = [c for c in X.columns if c not in cat_cols]
print(f"Features utilisées -> Numériques : {len(num_cols)} | Catégorielles : {len(cat_cols)}")

scaler_x = StandardScaler()
X[num_cols] = scaler_x.fit_transform(X[num_cols])
X_test_final[num_cols] = scaler_x.transform(X_test_final[num_cols])
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X[cat_cols] = encoder.fit_transform(X[cat_cols])
X_test_final[cat_cols] = encoder.transform(X_test_final[cat_cols])

def remap_unknown(x_array):
    x_array = x_array.astype(int)
    return x_array + 1
X[cat_cols] = remap_unknown(X[cat_cols].values)
X_test_final[cat_cols] = remap_unknown(X_test_final[cat_cols].values)
CAT_DIMS = [int(X[col].max() + 1) for col in cat_cols]
scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y)

X_num_tensor = torch.tensor(X[num_cols].values, dtype=torch.float32)
X_cat_tensor = torch.tensor(X[cat_cols].values, dtype=torch.long)
y_tensor = torch.tensor(y_scaled, dtype=torch.float32)

indices = np.arange(len(X))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42, shuffle=True)
X_train_num = X_num_tensor[train_idx]
X_train_cat = X_cat_tensor[train_idx]
y_train = y_tensor[train_idx]
X_val_num = X_num_tensor[val_idx]
X_val_cat = X_cat_tensor[val_idx]
y_val = y_tensor[val_idx]

val_ids = train_df.iloc[val_idx]['id'].values 
train_dataset = TensorDataset(X_train_num, X_train_cat, y_train)
val_dataset = TensorDataset(X_val_num, X_val_cat, y_val)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
X_test_num_tensor = torch.tensor(X_test_final[num_cols].values, dtype=torch.float32)
X_test_cat_tensor = torch.tensor(X_test_final[cat_cols].values, dtype=torch.long)
test_dataset = TensorDataset(X_test_num_tensor, X_test_cat_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Features numériques : 6
Features catégorielles : 7585


MemoryError: Unable to allocate 4.35 GiB for an array with shape (76905, 7585) and data type float64

## 3. Le Modèle

In [ ]:
class ReGLU(nn.Module):
    def forward(self, x):
        x_linear, x_gated = x.chunk(2, dim=-1)
        return x_linear * torch.relu(x_gated)

class FeatureTokenizer(nn.Module):
    def __init__(self, n_num, cat_dims, d_emb):
        super().__init__()
        self.num_weights = nn.Parameter(torch.Tensor(n_num, d_emb))
        self.num_biases = nn.Parameter(torch.Tensor(n_num, d_emb))
        nn.init.xavier_uniform_(self.num_weights)
        nn.init.zeros_(self.num_biases)
        self.cat_embeddings = nn.ModuleList([nn.Embedding(dim, d_emb) for dim in cat_dims])
        self.cat_biases = nn.ParameterList([nn.Parameter(torch.Tensor(d_emb)) for _ in cat_dims])
        for emb in self.cat_embeddings: nn.init.xavier_uniform_(emb.weight)
        for bias in self.cat_biases: nn.init.zeros_(bias)

        self.cls_token = nn.Parameter(torch.Tensor(1, 1, d_emb))
        nn.init.normal_(self.cls_token)

    def forward(self, x_num, x_cat):
        batch_size = x_num.shape[0]
        x_num = x_num.unsqueeze(2) 
        num_tokens = x_num * self.num_weights.unsqueeze(0) + self.num_biases.unsqueeze(0)

        cat_tokens = []
        for i, (emb, bias) in enumerate(zip(self.cat_embeddings, self.cat_biases)):
            token = emb(x_cat[:, i]) + bias
            cat_tokens.append(token.unsqueeze(1))
        if cat_tokens:
            cat_tokens = torch.cat(cat_tokens, dim=1)
            feature_tokens = torch.cat([num_tokens, cat_tokens], dim=1)
        else:
            feature_tokens = num_tokens
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        return torch.cat([cls_tokens, feature_tokens], dim=1)

class TransformerBlock(nn.Module):
    def __init__(self, d_emb, n_heads, ffn_factor, attn_dropout, ffn_dropout, resid_dropout):
        super().__init__()
        self.norm_attn = nn.LayerNorm(d_emb)
        self.attn = nn.MultiheadAttention(d_emb, n_heads, dropout=attn_dropout, batch_first=True)
        self.dropout_attn = nn.Dropout(resid_dropout)
        self.norm_ffn = nn.LayerNorm(d_emb)
        d_inner = int(d_emb * ffn_factor)
        self.ffn = nn.Sequential(
            nn.Linear(d_emb, d_inner * 2),
            ReGLU(),
            nn.Dropout(ffn_dropout),
            nn.Linear(d_inner, d_emb),
        )
        self.dropout_ffn = nn.Dropout(resid_dropout)
    def forward(self, x):
        x_norm = self.norm_attn(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + self.dropout_attn(attn_out)
        x_norm = self.norm_ffn(x)
        ffn_out = self.ffn(x_norm)
        x = x + self.dropout_ffn(ffn_out)
        return x

class FTTransformer(nn.Module):
    def __init__(self, n_num, cat_dims, d_emb, n_layers, n_heads, n_targets):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num, cat_dims, d_emb)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_emb, n_heads, FFN_FACTOR, ATTN_DROPOUT, FFN_DROPOUT, RESIDUAL_DROPOUT)
            for _ in range(n_layers)
        ])

        self.norm_final = nn.LayerNorm(d_emb)
        self.head = nn.Linear(d_emb, n_targets) 

    def forward(self, x_num, x_cat):
        x = self.tokenizer(x_num, x_cat)
        for block in self.blocks:
            x = block(x)
        x_cls = x[:, 0, :]
        x_cls = self.norm_final(x_cls)
        return self.head(x_cls)

## 4. Définition des critères

In [ ]:
def calculate_hit_rate(y_true, y_pred, delta=0.05):
    diff = np.abs(y_true - y_pred)
    hits = np.sum(diff <= delta)
    return hits / len(y_true)

def focused_satisfaction_loss(pred_sat, target_sat, delta=0.05):
    abs_error = torch.abs(pred_sat - target_sat)
    margin_violation = torch.relu(abs_error - delta)
    loss = torch.mean(margin_violation ** 2)
    mae_regularization = torch.mean(abs_error)

    return loss + 0.01 * mae_regularization

## 5. Entrainement

In [ ]:
model = FTTransformer(
    n_num=len(num_cols),
    cat_dims=CAT_DIMS,
    d_emb=D_EMB,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    n_targets=len(TARGETS)
).to(DEVICE)

no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], 'weight_decay': WEIGHT_DECAY},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
]
optimizer = optim.AdamW(optimizer_grouped_parameters, lr=LR)
criterion = nn.L1Loss()
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=5e-4,
    steps_per_epoch=len(train_loader),
    epochs=N_EPOCHS,
    pct_start=0.3  
)

best_hit_rate = -1.0
epochs_no_improve = 0
best_model_weights = None

for epoch in range(N_EPOCHS):
    model.train()
    train_loss = 0.0
    for xn, xc, y_batch in train_loader:
        xn, xc, y_batch = xn.to(DEVICE), xc.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        output = model(xn, xc)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item() * xn.size(0)
    
    avg_train_loss = train_loss / len(train_dataset)
    model.eval()
    val_preds = []
    val_trues = []
    with torch.no_grad():
        for xn, xc, y_batch in val_loader:
            xn, xc, y_batch = xn.to(DEVICE), xc.to(DEVICE), y_batch.to(DEVICE)
            output = model(xn, xc)
            val_preds.append(output.cpu().numpy())
            val_trues.append(y_batch.cpu().numpy())
    val_preds_arr = np.concatenate(val_preds, axis=0)
    val_trues_arr = np.concatenate(val_trues, axis=0)
    val_preds_original = scaler_y.inverse_transform(val_preds_arr)
    val_trues_original = scaler_y.inverse_transform(val_trues_arr)
    idx_sat = TARGETS.index('satisfaction')
    sat_pred = val_preds_original[:, idx_sat]
    sat_true = val_trues_original[:, idx_sat]
    current_hit_rate = calculate_hit_rate(sat_true, sat_pred, delta=0.05)
    print(f"Epoch {epoch+1:02d} | Train MSE: {avg_train_loss:.4f} | Val Hit Rate (Satisfaction): {current_hit_rate:.4%}")  

    if current_hit_rate > best_hit_rate:
        best_hit_rate = current_hit_rate
        best_model_weights = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Arrêt précoce ! Pas d'amélioration du Hit Rate depuis {PATIENCE} époques.")
            break

if best_model_weights:
    model.load_state_dict(best_model_weights)
    print(f"Meilleur modèle chargé (Hit Rate: {best_hit_rate:.4f})")

Début de l'entraînement MTL...
Epoch 1/150 | Train Loss: 5.4328 | Val hit rate: 0.4999
Epoch 2/150 | Train Loss: 5.4468 | Val hit rate: 0.4799
Epoch 3/150 | Train Loss: 5.3915 | Val hit rate: 0.4823
Epoch 4/150 | Train Loss: 5.3819 | Val hit rate: 0.4721
Epoch 5/150 | Train Loss: 5.3939 | Val hit rate: 0.4671
Epoch 6/150 | Train Loss: 5.3640 | Val hit rate: 0.5097
Epoch 7/150 | Train Loss: 5.3483 | Val hit rate: 0.5239
Epoch 8/150 | Train Loss: 5.4675 | Val hit rate: 0.5107
Epoch 9/150 | Train Loss: 5.2780 | Val hit rate: 0.4847
Epoch 10/150 | Train Loss: 5.2954 | Val hit rate: 0.5112
Epoch 11/150 | Train Loss: 5.4606 | Val hit rate: 0.4982
Epoch 12/150 | Train Loss: 5.3277 | Val hit rate: 0.4821
Epoch 13/150 | Train Loss: 5.3671 | Val hit rate: 0.4506
Epoch 14/150 | Train Loss: 5.2611 | Val hit rate: 0.5280
Epoch 15/150 | Train Loss: 5.2740 | Val hit rate: 0.4946
Epoch 16/150 | Train Loss: 5.2631 | Val hit rate: 0.5072
Epoch 17/150 | Train Loss: 5.3063 | Val hit rate: 0.4980
Epoch 18/

<All keys matched successfully>

## 6. Inférence et Sauvegarde

In [ ]:

model.eval()
val_preds_list = []
val_targets_list = []

with torch.no_grad():
    for xn, xc, y_true in val_loader:
        xn, xc = xn.to(DEVICE), xc.to(DEVICE)
        output = model(xn, xc)
        val_preds_list.append(output.cpu().numpy())
        val_targets_list.append(y_true.cpu().numpy())
val_preds_arr = np.concatenate(val_preds_list, axis=0)
val_targets_arr = np.concatenate(val_targets_list, axis=0)
val_preds_original = scaler_y.inverse_transform(val_preds_arr)
val_targets_original = scaler_y.inverse_transform(val_targets_arr)

stacking_df = pd.DataFrame({'id': val_ids})
for i, col in enumerate(TARGETS):
    stacking_df[f'pred_{col}'] = val_preds_original[:, i]
for i, col in enumerate(TARGETS):
    stacking_df[f'true_{col}'] = val_targets_original[:, i]
os.makedirs('stacking_data', exist_ok=True)
stacking_file = 'stacking_data/ft_transformer_val_preds.csv'
stacking_df.to_csv(stacking_file, index=False)
print(f"Fichier de stacking sauvegardé : {stacking_file}")
print(f"Dimensions : {stacking_df.shape}")

In [ ]:
model.eval()
preds_list = []
with torch.no_grad():
    for xn, xc in test_loader:
        xn, xc = xn.to(DEVICE), xc.to(DEVICE)
        output = model(xn, xc)
        preds_list.append(output.cpu().numpy())
final_preds = scaler_y.inverse_transform(np.concatenate(preds_list, axis=0))
submission_df = pd.DataFrame({'id': test_ids})
for i, col in enumerate(TARGETS):
    submission_df[col] = final_preds[:, i]
output_file = 'outputs/ft_transformer_L1Loss_Scheduler.csv'
os.makedirs('outputs', exist_ok=True)
submission_df.to_csv(output_file, index=False)
print(f"\nFichier sauvegardé : {output_file}")


=== RÉSULTATS FINAUX (MTL) ===
Comparaison avec LightGBM sur le hit rate @ 0.05 (Valeurs Réelles)
Hit Rate satisfaction (MTL): 0.6357
